In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Physics Engine Setup ---
x = np.linspace(-5, 5, 500)
dx = x[1] - x[0]

# Pre-calculate base wave functions for speed
psi_L_base = np.exp(-(x + 2.2)**2 / (2 * 0.4**2))
psi_R_base = np.exp(-(x - 2.2)**2 / (2 * 0.4**2))
psi_L_base /= np.sqrt(np.sum(psi_L_base**2) * dx)
psi_R_base /= np.sqrt(np.sum(psi_R_base**2) * dx)

# Global state to track if the electron has "collapsed"
state = {'measurement': 0} # 0: Superposition, 1: Left, 2: Right

def get_prob(t, deco_rate, outcome):
    if outcome == 1: return psi_L_base**2
    if outcome == 2: return psi_R_base**2

    # Superposition logic with decoherence
    coherence = np.exp(-deco_rate * t)
    # The 'cos(5t)' creates the quantum phase oscillation
    prob = (psi_L_base**2 + psi_R_base**2) + 2 * psi_L_base * psi_R_base * coherence * np.cos(5*t)
    return prob / np.sum(prob * dx)

# --- 2. UI Elements ---
slider_p = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Pressure (P):')
slider_t = widgets.IntSlider(value=273, min=10, max=500, description='Temp (T):')
slider_time = widgets.FloatSlider(value=0.0, min=0.0, max=50.0, step=0.5, description='Time (t):')
btn_measure = widgets.Button(description='MEASURE!', button_style='warning', tooltip='Force the wave to collapse')
btn_reset = widgets.Button(description='Reset Quantum State', button_style='info')
output = widgets.Output()

# --- 3. Interaction Logic ---
def update_plot(change=None):
    with output:
        clear_output(wait=True)

        P = slider_p.value
        T = slider_t.value
        time = slider_time.value

        # Calculate decoherence rate based on gas properties
        deco_rate = (P / np.sqrt(T)) * 0.5

        # If time is reset to 0, reset the measurement result
        if time == 0: state['measurement'] = 0

        prob = get_prob(time, deco_rate, state['measurement'])

        # Create Plot
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(x, prob, lw=3, color='dodgerblue' if state['measurement']==0 else 'crimson')

        # Draw Potential Well Background
        ax_pot = ax.twinx()
        ax_pot.plot(x, 0.5*x**4 - 5*x**2, 'k--', alpha=0.1)
        ax_pot.set_yticks([])

        # Styling
        title = "State: Quantum Superposition" if state['measurement'] == 0 else "State: CLASSICAL (Collapsed)"
        ax.set_title(f"{title}\nDecoherence Rate: {deco_rate:.3f}", fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.5)
        ax.set_xlabel("Position in Nano-meters")
        ax.set_ylabel("Probability Density")
        ax.grid(True, alpha=0.2)
        plt.show()

def on_measure_clicked(b):
    if state['measurement'] != 0: return

    # Calculate current probabilities to decide outcome
    deco_rate = (slider_p.value / np.sqrt(slider_t.value)) * 0.5
    current_p = get_prob(slider_time.value, deco_rate, 0)

    mid = len(x)//2
    p_left = np.sum(current_p[:mid])
    p_right = np.sum(current_p[mid:])

    # The Born Rule: Random choice based on wave function shape
    state['measurement'] = np.random.choice([1, 2], p=[p_left/(p_left+p_right), p_right/(p_left+p_right)])
    update_plot()

def on_reset_clicked(b):
    state['measurement'] = 0
    slider_time.value = 0
    update_plot()

# Link sliders to the update function
slider_p.observe(update_plot, names='value')
slider_t.observe(update_plot, names='value')
slider_time.observe(update_plot, names='value')
btn_measure.on_click(on_measure_clicked)
btn_reset.on_click(on_reset_clicked)

# --- 4. Display Everything ---
# Arrange sliders in a nice layout
controls = widgets.VBox([slider_p, slider_t, slider_time, widgets.HBox([btn_measure, btn_reset])])
display(widgets.HBox([controls, output]))

# Trigger the first frame
update_plot()